In [1]:
from pathlib import Path
import json

import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("..")

validation_dir = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "validation"
)

clean_frames = np.load(
    validation_dir
    / "clean_frames.npy"
)

injected_frames = np.load(
    validation_dir
    / "injected_frames.npy"
)

with open(
    validation_dir / "metadata.json"
) as file:
    metadata = json.load(file)

device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)
print("Clean:", clean_frames.shape)
print("Injected:", injected_frames.shape)
print("Metadata:", len(metadata))
print("Metadata keys:", metadata[0].keys())

Device: mps
Clean: (700, 16, 4096)
Injected: (700, 16, 4096)
Metadata: 700
Metadata keys: dict_keys(['case_id', 'frame_index', 'observation_id', 'source_name', 'frame_index_within_observation', 'morphology', 'robust_snr', 'bandwidth_bins', 'frequency_bin', 'repeat'])


In [2]:
from deepseti.models.vit_mae import ViTMAE


def build_vit_mae():
    return ViTMAE(
        frame_shape=(16, 4096),
        patch_size=(4, 64),
        embed_dim=128,
        encoder_depth=4,
        encoder_heads=4,
        encoder_mlp_dim=256,
        decoder_dim=64,
        decoder_depth=2,
        decoder_heads=4,
        decoder_mlp_dim=128,
        mask_ratio=0.75,
    )


checkpoint_dir = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
)

epoch100_checkpoint = torch.load(
    checkpoint_dir
    / "vit_mae_epoch100_training_state.pt",
    map_location=device,
)

epoch125_checkpoint = torch.load(
    checkpoint_dir
    / "vit_mae_epoch125_training_state.pt",
    map_location=device,
)


model100 = build_vit_mae().to(device)
model125 = build_vit_mae().to(device)


model100.load_state_dict(
    epoch100_checkpoint["model_state_dict"]
)

model125.load_state_dict(
    epoch125_checkpoint["model_state_dict"]
)


model100.eval()
model125.eval()


print(
    "Epoch 100:",
    epoch100_checkpoint["epoch"],
)

print(
    "Epoch 125:",
    epoch125_checkpoint["epoch"],
)

print(
    "Epoch 100 parameters:",
    sum(p.numel() for p in model100.parameters()),
)

print(
    "Epoch 125 parameters:",
    sum(p.numel() for p in model125.parameters()),
)

/Users/parthshringarpure/deepseti/src/deepseti/models/vit_mae.py:429: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(
/Users/parthshringarpure/deepseti/src/deepseti/models/vit_mae.py:519: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.decoder = nn.TransformerEncoder(


Epoch 100: 100
Epoch 125: 125
Epoch 100 parameters: 668160
Epoch 125 parameters: 668160


In [3]:
from deepseti.models.vit_mae import ViTMAE


def build_vit_mae():
    return ViTMAE(
        frame_shape=(16, 4096),
        patch_size=(4, 64),
        embed_dim=128,
        encoder_depth=4,
        encoder_heads=4,
        encoder_mlp_dim=256,
        decoder_dim=64,
        decoder_depth=2,
        decoder_heads=4,
        decoder_mlp_dim=128,
        mask_ratio=0.75,
    )


checkpoint_dir = (
    PROJECT_ROOT
    / "results"
    / "checkpoints"
)

epoch100_checkpoint = torch.load(
    checkpoint_dir
    / "vit_mae_epoch100_training_state.pt",
    map_location=device,
)

epoch125_checkpoint = torch.load(
    checkpoint_dir
    / "vit_mae_epoch125_training_state.pt",
    map_location=device,
)


model100 = build_vit_mae().to(device)
model125 = build_vit_mae().to(device)


model100.load_state_dict(
    epoch100_checkpoint["model_state_dict"]
)

model125.load_state_dict(
    epoch125_checkpoint["model_state_dict"]
)


model100.eval()
model125.eval()


print(
    "Epoch 100:",
    epoch100_checkpoint["epoch"],
)

print(
    "Epoch 125:",
    epoch125_checkpoint["epoch"],
)

print(
    "Epoch 100 parameters:",
    sum(p.numel() for p in model100.parameters()),
)

print(
    "Epoch 125 parameters:",
    sum(p.numel() for p in model125.parameters()),
)

Epoch 100: 100
Epoch 125: 125
Epoch 100 parameters: 668160
Epoch 125 parameters: 668160


In [4]:
from deepseti.models.vit_mae import patchify


def make_shared_mask(
    batch_size,
    num_patches=256,
    mask_ratio=0.75,
    seed=42,
):
    generator = torch.Generator()
    generator.manual_seed(seed)

    noise = torch.rand(
        batch_size,
        num_patches,
        generator=generator,
    )

    ids_shuffle = torch.argsort(
        noise,
        dim=1,
    )

    ids_restore = torch.argsort(
        ids_shuffle,
        dim=1,
    )

    num_visible = int(
        num_patches
        * (1.0 - mask_ratio)
    )

    ids_keep = ids_shuffle[
        :,
        :num_visible,
    ]

    mask = torch.ones(
        batch_size,
        num_patches,
    )

    mask[:, :num_visible] = 0

    mask = torch.gather(
        mask,
        dim=1,
        index=ids_restore,
    )

    return (
        ids_keep,
        ids_restore,
        mask,
    )


def forward_with_fixed_mask(
    model,
    x,
    ids_keep,
    ids_restore,
    mask,
):
    target_patches = patchify(
        x,
        patch_size=(4, 64),
    )

    tokens = model.patch_embedding(x)
    tokens = model.position_embedding(tokens)

    embed_dim = tokens.shape[-1]

    ids_keep = ids_keep.to(x.device)
    ids_restore = ids_restore.to(x.device)
    mask = mask.to(x.device)

    visible_tokens = torch.gather(
        tokens,
        dim=1,
        index=ids_keep.unsqueeze(-1).expand(
            -1,
            -1,
            embed_dim,
        ),
    )

    encoded_tokens = model.encoder(
        visible_tokens
    )

    predicted_patches = model.decoder(
        encoded_tokens,
        ids_restore,
    )

    return (
        predicted_patches,
        target_patches,
        mask,
    )

In [5]:
batch_size = 8

clean_batch = torch.tensor(
    clean_frames[:batch_size],
    dtype=torch.float32,
).unsqueeze(1).to(device)

injected_batch = torch.tensor(
    injected_frames[:batch_size],
    dtype=torch.float32,
).unsqueeze(1).to(device)


ids_keep, ids_restore, shared_mask = make_shared_mask(
    batch_size=batch_size,
    seed=42,
)


with torch.no_grad():

    clean_pred, clean_target, clean_mask = (
        forward_with_fixed_mask(
            model125,
            clean_batch,
            ids_keep,
            ids_restore,
            shared_mask,
        )
    )

    injected_pred, injected_target, injected_mask = (
        forward_with_fixed_mask(
            model125,
            injected_batch,
            ids_keep,
            ids_restore,
            shared_mask,
        )
    )


print(
    "Clean prediction:",
    clean_pred.shape,
)

print(
    "Injected prediction:",
    injected_pred.shape,
)

print(
    "Same masks:",
    torch.equal(
        clean_mask,
        injected_mask,
    ),
)

print(
    "Masked patches:",
    int(clean_mask[0].sum().item()),
)

Clean prediction: torch.Size([8, 256, 256])
Injected prediction: torch.Size([8, 256, 256])
Same masks: True
Masked patches: 192


In [6]:
def masked_frame_scores(
    predicted_patches,
    target_patches,
    mask,
):
    """
    Return one masked reconstruction-error
    score per spectrogram frame.
    """

    # MSE inside each patch
    patch_error = (
        (
            predicted_patches
            - target_patches
        ) ** 2
    ).mean(dim=-1)

    # Use only masked patches
    frame_scores = (
        patch_error
        * mask
    ).sum(dim=1) / mask.sum(dim=1)

    return frame_scores


clean_scores = masked_frame_scores(
    clean_pred,
    clean_target,
    clean_mask,
)

injected_scores = masked_frame_scores(
    injected_pred,
    injected_target,
    injected_mask,
)


print("Clean scores:")
print(clean_scores.cpu().numpy())

print()

print("Injected scores:")
print(injected_scores.cpu().numpy())

print()

print(
    "Mean clean:",
    clean_scores.mean().item(),
)

print(
    "Mean injected:",
    injected_scores.mean().item(),
)

print(
    "Injected higher in:",
    int(
        (
            injected_scores
            > clean_scores
        ).sum().item()
    ),
    "/",
    batch_size,
)

Clean scores:
[0.3143216  0.13701557 0.14724676 0.1690752  0.12944974 4.575603
 0.23283261 3.33693   ]

Injected scores:
[0.31491932 0.13732877 0.1475571  0.16889407 0.12956643 4.578499
 0.2326868  3.3373673 ]

Mean clean: 1.1303093433380127
Mean injected: 1.1308523416519165
Injected higher in: 6 / 8


In [7]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
)


def score_paired_dataset(
    model,
    clean_frames,
    injected_frames,
    batch_size=16,
    base_seed=42,
):
    clean_all = []
    injected_all = []

    model.eval()

    with torch.no_grad():

        for start in range(
            0,
            len(clean_frames),
            batch_size,
        ):
            stop = min(
                start + batch_size,
                len(clean_frames),
            )

            clean_batch = torch.tensor(
                clean_frames[start:stop],
                dtype=torch.float32,
            ).unsqueeze(1).to(device)

            injected_batch = torch.tensor(
                injected_frames[start:stop],
                dtype=torch.float32,
            ).unsqueeze(1).to(device)

            current_batch_size = (
                stop - start
            )

            # Different deterministic masks
            # across batches, but shared between
            # clean and injected pairs.
            (
                ids_keep,
                ids_restore,
                shared_mask,
            ) = make_shared_mask(
                batch_size=current_batch_size,
                seed=base_seed + start,
            )

            clean_pred, clean_target, clean_mask = (
                forward_with_fixed_mask(
                    model,
                    clean_batch,
                    ids_keep,
                    ids_restore,
                    shared_mask,
                )
            )

            (
                injected_pred,
                injected_target,
                injected_mask,
            ) = forward_with_fixed_mask(
                model,
                injected_batch,
                ids_keep,
                ids_restore,
                shared_mask,
            )

            clean_scores = masked_frame_scores(
                clean_pred,
                clean_target,
                clean_mask,
            )

            injected_scores = masked_frame_scores(
                injected_pred,
                injected_target,
                injected_mask,
            )

            clean_all.append(
                clean_scores.cpu().numpy()
            )

            injected_all.append(
                injected_scores.cpu().numpy()
            )

    return (
        np.concatenate(clean_all),
        np.concatenate(injected_all),
    )

In [8]:
clean_scores_125, injected_scores_125 = (
    score_paired_dataset(
        model125,
        clean_frames,
        injected_frames,
        batch_size=16,
        base_seed=42,
    )
)

labels = np.concatenate(
    [
        np.zeros(len(clean_scores_125)),
        np.ones(len(injected_scores_125)),
    ]
)

scores = np.concatenate(
    [
        clean_scores_125,
        injected_scores_125,
    ]
)

auroc = roc_auc_score(
    labels,
    scores,
)

auprc = average_precision_score(
    labels,
    scores,
)

paired_win_rate = np.mean(
    injected_scores_125
    > clean_scores_125
)

print("Epoch 125 — single mask")
print(f"AUROC: {auroc:.6f}")
print(f"AUPRC: {auprc:.6f}")
print(
    "Injected > clean:",
    f"{paired_win_rate:.3%}",
)
print(
    "Mean clean:",
    clean_scores_125.mean(),
)
print(
    "Mean injected:",
    injected_scores_125.mean(),
)

Epoch 125 — single mask
AUROC: 0.536171
AUPRC: 0.520975
Injected > clean: 83.143%
Mean clean: 1.4685674
Mean injected: 1.5219817


In [9]:
clean_scores_100, injected_scores_100 = (
    score_paired_dataset(
        model100,
        clean_frames,
        injected_frames,
        batch_size=16,
        base_seed=42,
    )
)

labels_100 = np.concatenate(
    [
        np.zeros(len(clean_scores_100)),
        np.ones(len(injected_scores_100)),
    ]
)

scores_100 = np.concatenate(
    [
        clean_scores_100,
        injected_scores_100,
    ]
)

auroc_100 = roc_auc_score(
    labels_100,
    scores_100,
)

auprc_100 = average_precision_score(
    labels_100,
    scores_100,
)

paired_win_rate_100 = np.mean(
    injected_scores_100
    > clean_scores_100
)

print("Epoch 100 — single mask")
print(f"AUROC: {auroc_100:.6f}")
print(f"AUPRC: {auprc_100:.6f}")
print(
    "Injected > clean:",
    f"{paired_win_rate_100:.3%}",
)
print(
    "Mean clean:",
    clean_scores_100.mean(),
)
print(
    "Mean injected:",
    injected_scores_100.mean(),
)

Epoch 100 — single mask
AUROC: 0.524092
AUPRC: 0.513170
Injected > clean: 70.000%
Mean clean: 2.5035102
Mean injected: 2.5527585


In [10]:
def masked_topk_scores(
    predicted_patches,
    target_patches,
    mask,
    top_fraction,
):
    """
    Score each frame using the largest reconstruction
    errors among masked pixels only.

    top_fraction is defined relative to the full
    16 x 4096 frame for comparability with baselines.
    """

    pixel_error = (
        predicted_patches
        - target_patches
    ) ** 2

    batch, num_patches, patch_dim = (
        pixel_error.shape
    )

    full_frame_pixels = (
        num_patches * patch_dim
    )

    k = max(
        1,
        int(
            full_frame_pixels
            * top_fraction
        ),
    )

    scores = []

    for i in range(batch):

        masked_patch_error = pixel_error[
            i,
            mask[i].bool(),
        ]

        masked_pixel_error = (
            masked_patch_error.reshape(-1)
        )

        top_values = torch.topk(
            masked_pixel_error,
            k=k,
        ).values

        scores.append(
            top_values.mean()
        )

    return torch.stack(scores)

In [11]:
def score_paired_dataset_topk(
    model,
    clean_frames,
    injected_frames,
    top_fraction,
    batch_size=16,
    base_seed=42,
):
    clean_all = []
    injected_all = []

    model.eval()

    with torch.no_grad():

        for start in range(
            0,
            len(clean_frames),
            batch_size,
        ):
            stop = min(
                start + batch_size,
                len(clean_frames),
            )

            clean_batch = torch.tensor(
                clean_frames[start:stop],
                dtype=torch.float32,
            ).unsqueeze(1).to(device)

            injected_batch = torch.tensor(
                injected_frames[start:stop],
                dtype=torch.float32,
            ).unsqueeze(1).to(device)

            current_batch_size = (
                stop - start
            )

            (
                ids_keep,
                ids_restore,
                shared_mask,
            ) = make_shared_mask(
                batch_size=current_batch_size,
                seed=base_seed + start,
            )

            clean_pred, clean_target, clean_mask = (
                forward_with_fixed_mask(
                    model,
                    clean_batch,
                    ids_keep,
                    ids_restore,
                    shared_mask,
                )
            )

            injected_pred, injected_target, injected_mask = (
                forward_with_fixed_mask(
                    model,
                    injected_batch,
                    ids_keep,
                    ids_restore,
                    shared_mask,
                )
            )

            clean_score = masked_topk_scores(
                clean_pred,
                clean_target,
                clean_mask,
                top_fraction,
            )

            injected_score = masked_topk_scores(
                injected_pred,
                injected_target,
                injected_mask,
                top_fraction,
            )

            clean_all.append(
                clean_score.cpu().numpy()
            )

            injected_all.append(
                injected_score.cpu().numpy()
            )

    return (
        np.concatenate(clean_all),
        np.concatenate(injected_all),
    )

In [12]:
top_fractions = [
    0.00025,
    0.00050,
    0.001,
    0.002,
    0.005,
    0.010,
]

topk_results = []

for fraction in top_fractions:

    clean_score, injected_score = (
        score_paired_dataset_topk(
            model125,
            clean_frames,
            injected_frames,
            top_fraction=fraction,
            batch_size=16,
            base_seed=42,
        )
    )

    labels = np.concatenate([
        np.zeros(len(clean_score)),
        np.ones(len(injected_score)),
    ])

    scores = np.concatenate([
        clean_score,
        injected_score,
    ])

    auroc = roc_auc_score(
        labels,
        scores,
    )

    auprc = average_precision_score(
        labels,
        scores,
    )

    win_rate = np.mean(
        injected_score > clean_score
    )

    topk_results.append(
        (
            fraction,
            auroc,
            auprc,
            win_rate,
        )
    )

    print(
        f"{fraction:.5f} | "
        f"AUROC {auroc:.4f} | "
        f"AUPRC {auprc:.4f} | "
        f"paired {win_rate:.1%}"
    )

0.00025 | AUROC 0.5507 | AUPRC 0.5242 | paired 59.0%
0.00050 | AUROC 0.5549 | AUPRC 0.5268 | paired 64.4%
0.00100 | AUROC 0.5589 | AUPRC 0.5292 | paired 70.4%
0.00200 | AUROC 0.5603 | AUPRC 0.5300 | paired 73.0%
0.00500 | AUROC 0.5595 | AUPRC 0.5298 | paired 75.9%
0.01000 | AUROC 0.5593 | AUPRC 0.5301 | paired 77.9%


In [13]:
def score_paired_dataset_multimask(
    model,
    clean_frames,
    injected_frames,
    top_fraction=0.002,
    num_masks=4,
    batch_size=16,
    base_seed=42,
):
    clean_sum = np.zeros(
        len(clean_frames),
        dtype=np.float64,
    )

    injected_sum = np.zeros(
        len(injected_frames),
        dtype=np.float64,
    )

    model.eval()

    with torch.no_grad():

        for mask_index in range(num_masks):

            for start in range(
                0,
                len(clean_frames),
                batch_size,
            ):
                stop = min(
                    start + batch_size,
                    len(clean_frames),
                )

                clean_batch = torch.tensor(
                    clean_frames[start:stop],
                    dtype=torch.float32,
                ).unsqueeze(1).to(device)

                injected_batch = torch.tensor(
                    injected_frames[start:stop],
                    dtype=torch.float32,
                ).unsqueeze(1).to(device)

                current_batch_size = (
                    stop - start
                )

                # Different deterministic mask
                # for every repeat.
                seed = (
                    base_seed
                    + start
                    + mask_index * 100_000
                )

                (
                    ids_keep,
                    ids_restore,
                    shared_mask,
                ) = make_shared_mask(
                    batch_size=current_batch_size,
                    seed=seed,
                )

                (
                    clean_pred,
                    clean_target,
                    clean_mask,
                ) = forward_with_fixed_mask(
                    model,
                    clean_batch,
                    ids_keep,
                    ids_restore,
                    shared_mask,
                )

                (
                    injected_pred,
                    injected_target,
                    injected_mask,
                ) = forward_with_fixed_mask(
                    model,
                    injected_batch,
                    ids_keep,
                    ids_restore,
                    shared_mask,
                )

                clean_score = masked_topk_scores(
                    clean_pred,
                    clean_target,
                    clean_mask,
                    top_fraction,
                )

                injected_score = masked_topk_scores(
                    injected_pred,
                    injected_target,
                    injected_mask,
                    top_fraction,
                )

                clean_sum[start:stop] += (
                    clean_score.cpu().numpy()
                )

                injected_sum[start:stop] += (
                    injected_score.cpu().numpy()
                )

    return (
        clean_sum / num_masks,
        injected_sum / num_masks,
    )

In [14]:
multimask_results = []

for num_masks in [1, 4, 8]:

    clean_score, injected_score = (
        score_paired_dataset_multimask(
            model125,
            clean_frames,
            injected_frames,
            top_fraction=0.002,
            num_masks=num_masks,
            batch_size=16,
            base_seed=42,
        )
    )

    labels = np.concatenate([
        np.zeros(len(clean_score)),
        np.ones(len(injected_score)),
    ])

    scores = np.concatenate([
        clean_score,
        injected_score,
    ])

    auroc = roc_auc_score(
        labels,
        scores,
    )

    auprc = average_precision_score(
        labels,
        scores,
    )

    paired = np.mean(
        injected_score > clean_score
    )

    multimask_results.append(
        (
            num_masks,
            auroc,
            auprc,
            paired,
        )
    )

    print(
        f"{num_masks:2d} mask(s) | "
        f"AUROC {auroc:.4f} | "
        f"AUPRC {auprc:.4f} | "
        f"paired {paired:.1%}"
    )

 1 mask(s) | AUROC 0.5603 | AUPRC 0.5300 | paired 73.0%
 4 mask(s) | AUROC 0.5602 | AUPRC 0.5293 | paired 74.9%
 8 mask(s) | AUROC 0.5600 | AUPRC 0.5289 | paired 76.3%


In [15]:
clean_topk_100, injected_topk_100 = (
    score_paired_dataset_topk(
        model100,
        clean_frames,
        injected_frames,
        top_fraction=0.002,
        batch_size=16,
        base_seed=42,
    )
)

labels_100 = np.concatenate([
    np.zeros(len(clean_topk_100)),
    np.ones(len(injected_topk_100)),
])

scores_100 = np.concatenate([
    clean_topk_100,
    injected_topk_100,
])

auroc_100_topk = roc_auc_score(
    labels_100,
    scores_100,
)

auprc_100_topk = average_precision_score(
    labels_100,
    scores_100,
)

paired_100_topk = np.mean(
    injected_topk_100
    > clean_topk_100
)

print("Epoch 100 — top fraction 0.002")
print(f"AUROC: {auroc_100_topk:.4f}")
print(f"AUPRC: {auprc_100_topk:.4f}")
print(f"Paired: {paired_100_topk:.1%}")

print()
print("Epoch 125 — top fraction 0.002")
print("AUROC: 0.5603")
print("AUPRC: 0.5300")
print("Paired: 73.0%")

Epoch 100 — top fraction 0.002
AUROC: 0.5511
AUPRC: 0.5246
Paired: 64.7%

Epoch 125 — top fraction 0.002
AUROC: 0.5603
AUPRC: 0.5300
Paired: 73.0%


In [16]:
# Recompute and store our FINAL selected validation scores.

clean_selected_125, injected_selected_125 = (
    score_paired_dataset_topk(
        model125,
        clean_frames,
        injected_frames,
        top_fraction=0.002,
        batch_size=16,
        base_seed=42,
    )
)

print(
    "Stored validation scores:",
    clean_selected_125.shape,
    injected_selected_125.shape,
)

Stored validation scores: (700,) (700,)


In [17]:
# Recompute and store our FINAL selected validation scores.

clean_selected_125, injected_selected_125 = (
    score_paired_dataset_topk(
        model125,
        clean_frames,
        injected_frames,
        top_fraction=0.002,
        batch_size=16,
        base_seed=42,
    )
)

print(
    "Stored validation scores:",
    clean_selected_125.shape,
    injected_selected_125.shape,
)

Stored validation scores: (700,) (700,)


In [18]:
def subset_metrics(indices):
    clean_subset = clean_selected_125[indices]
    injected_subset = injected_selected_125[indices]

    labels = np.concatenate([
        np.zeros(len(clean_subset)),
        np.ones(len(injected_subset)),
    ])

    scores = np.concatenate([
        clean_subset,
        injected_subset,
    ])

    return {
        "auroc": roc_auc_score(
            labels,
            scores,
        ),
        "auprc": average_precision_score(
            labels,
            scores,
        ),
        "paired": np.mean(
            injected_subset
            > clean_subset
        ),
        "n": len(indices),
    }


# -------------------------
# Performance by SNR
# -------------------------

snr_values = sorted({
    item["robust_snr"]
    for item in metadata
})

print("BY SNR")
print()

for snr in snr_values:

    indices = np.array([
        i
        for i, item in enumerate(metadata)
        if item["robust_snr"] == snr
    ])

    result = subset_metrics(indices)

    print(
        f"SNR {snr:>4} | "
        f"n={result['n']:3d} | "
        f"AUROC {result['auroc']:.3f} | "
        f"AUPRC {result['auprc']:.3f} | "
        f"paired {result['paired']:.1%}"
    )


# -------------------------
# Performance by morphology
# -------------------------

morphologies = sorted({
    item["morphology"]
    for item in metadata
})

print()
print("BY MORPHOLOGY")
print()

for morphology in morphologies:

    indices = np.array([
        i
        for i, item in enumerate(metadata)
        if item["morphology"] == morphology
    ])

    result = subset_metrics(indices)

    print(
        f"{morphology:22s} | "
        f"n={result['n']:3d} | "
        f"AUROC {result['auroc']:.3f} | "
        f"AUPRC {result['auprc']:.3f} | "
        f"paired {result['paired']:.1%}"
    )

BY SNR

SNR  0.5 | n=140 | AUROC 0.501 | AUPRC 0.508 | paired 57.9%
SNR  1.0 | n=140 | AUROC 0.505 | AUPRC 0.504 | paired 59.3%
SNR  2.0 | n=140 | AUROC 0.518 | AUPRC 0.509 | paired 69.3%
SNR  5.0 | n=140 | AUROC 0.596 | AUPRC 0.548 | paired 83.6%
SNR 10.0 | n=140 | AUROC 0.682 | AUPRC 0.599 | paired 95.0%

BY MORPHOLOGY

burst                  | n=225 | AUROC 0.581 | AUPRC 0.548 | paired 72.0%
intermittent           | n= 75 | AUROC 0.513 | AUPRC 0.518 | paired 64.0%
linear_drift           | n=150 | AUROC 0.553 | AUPRC 0.528 | paired 76.0%
multiple_carriers      | n= 75 | AUROC 0.584 | AUPRC 0.549 | paired 77.3%
nonlinear_drift        | n=100 | AUROC 0.546 | AUPRC 0.527 | paired 74.0%
stationary_narrowband  | n= 75 | AUROC 0.573 | AUPRC 0.542 | paired 73.3%


In [19]:
test_dir = (
    PROJECT_ROOT
    / "data"
    / "synthetic"
    / "test"
)

test_clean_frames = np.load(
    test_dir / "clean_frames.npy"
)

test_injected_frames = np.load(
    test_dir / "injected_frames.npy"
)

with open(
    test_dir / "metadata.json"
) as file:
    test_metadata = json.load(file)

print(
    "Clean:",
    test_clean_frames.shape,
)

print(
    "Injected:",
    test_injected_frames.shape,
)

print(
    "Metadata:",
    len(test_metadata),
)

Clean: (700, 16, 4096)
Injected: (700, 16, 4096)
Metadata: 700


In [20]:
test_clean_scores, test_injected_scores = (
    score_paired_dataset_topk(
        model125,
        test_clean_frames,
        test_injected_frames,
        top_fraction=0.002,
        batch_size=16,
        base_seed=42,
    )
)

test_labels = np.concatenate([
    np.zeros(len(test_clean_scores)),
    np.ones(len(test_injected_scores)),
])

test_scores = np.concatenate([
    test_clean_scores,
    test_injected_scores,
])

test_auroc = roc_auc_score(
    test_labels,
    test_scores,
)

test_auprc = average_precision_score(
    test_labels,
    test_scores,
)

test_paired = np.mean(
    test_injected_scores
    > test_clean_scores
)

print("ViT-MAE development-test")
print(f"AUROC: {test_auroc:.6f}")
print(f"AUPRC: {test_auprc:.6f}")
print(
    "Injected > clean:",
    f"{test_paired:.3%}",
)
print(
    "Mean clean:",
    test_clean_scores.mean(),
)
print(
    "Mean injected:",
    test_injected_scores.mean(),
)


ViT-MAE development-test
AUROC: 0.578545
AUPRC: 0.550394
Injected > clean: 74.571%
Mean clean: 90.14589
Mean injected: 99.400986


In [21]:
def test_subset_metrics(indices):
    clean_subset = test_clean_scores[indices]
    injected_subset = test_injected_scores[indices]

    labels = np.concatenate([
        np.zeros(len(clean_subset)),
        np.ones(len(injected_subset)),
    ])

    scores = np.concatenate([
        clean_subset,
        injected_subset,
    ])

    return {
        "auroc": roc_auc_score(labels, scores),
        "auprc": average_precision_score(labels, scores),
        "paired": np.mean(
            injected_subset > clean_subset
        ),
        "n": len(indices),
    }


print("BY SNR")
print()

snr_values = sorted({
    item["robust_snr"]
    for item in test_metadata
})

for snr in snr_values:

    indices = np.array([
        i
        for i, item in enumerate(test_metadata)
        if item["robust_snr"] == snr
    ])

    result = test_subset_metrics(indices)

    print(
        f"SNR {snr:>4} | "
        f"n={result['n']:3d} | "
        f"AUROC {result['auroc']:.3f} | "
        f"AUPRC {result['auprc']:.3f} | "
        f"paired {result['paired']:.1%}"
    )


print()
print("BY MORPHOLOGY")
print()

morphologies = sorted({
    item["morphology"]
    for item in test_metadata
})

for morphology in morphologies:

    indices = np.array([
        i
        for i, item in enumerate(test_metadata)
        if item["morphology"] == morphology
    ])

    result = test_subset_metrics(indices)

    print(
        f"{morphology:22s} | "
        f"n={result['n']:3d} | "
        f"AUROC {result['auroc']:.3f} | "
        f"AUPRC {result['auprc']:.3f} | "
        f"paired {result['paired']:.1%}"
    )

BY SNR

SNR  0.5 | n=140 | AUROC 0.501 | AUPRC 0.504 | paired 55.0%
SNR  1.0 | n=140 | AUROC 0.506 | AUPRC 0.503 | paired 65.7%
SNR  2.0 | n=140 | AUROC 0.525 | AUPRC 0.507 | paired 70.0%
SNR  5.0 | n=140 | AUROC 0.619 | AUPRC 0.562 | paired 84.3%
SNR 10.0 | n=140 | AUROC 0.743 | AUPRC 0.677 | paired 97.9%

BY MORPHOLOGY

burst                  | n=225 | AUROC 0.602 | AUPRC 0.593 | paired 77.3%
intermittent           | n= 75 | AUROC 0.524 | AUPRC 0.523 | paired 69.3%
linear_drift           | n=150 | AUROC 0.570 | AUPRC 0.538 | paired 74.7%
multiple_carriers      | n= 75 | AUROC 0.598 | AUPRC 0.568 | paired 77.3%
nonlinear_drift        | n=100 | AUROC 0.560 | AUPRC 0.534 | paired 71.0%
stationary_narrowband  | n= 75 | AUROC 0.600 | AUPRC 0.558 | paired 73.3%


In [22]:
results_dir = (
    PROJECT_ROOT
    / "results"
    / "metrics"
)

results_dir.mkdir(
    parents=True,
    exist_ok=True,
)

np.save(
    results_dir
    / "vit_mae_dev_test_clean_scores.npy",
    test_clean_scores,
)

np.save(
    results_dir
    / "vit_mae_dev_test_injected_scores.npy",
    test_injected_scores,
)

vit_mae_summary = {
    "checkpoint_epoch": 125,
    "patch_size": [4, 64],
    "mask_ratio": 0.75,
    "num_inference_masks": 1,
    "score": "masked_top_error",
    "top_fraction": 0.002,
    "development_test": {
        "auroc": float(test_auroc),
        "auprc": float(test_auprc),
        "paired_win_rate": float(test_paired),
    },
}

with open(
    results_dir
    / "vit_mae_dev_test_summary.json",
    "w",
) as file:
    json.dump(
        vit_mae_summary,
        file,
        indent=2,
    )

print(
    json.dumps(
        vit_mae_summary,
        indent=2,
    )
)

{
  "checkpoint_epoch": 125,
  "patch_size": [
    4,
    64
  ],
  "mask_ratio": 0.75,
  "num_inference_masks": 1,
  "score": "masked_top_error",
  "top_fraction": 0.002,
  "development_test": {
    "auroc": 0.5785448979591836,
    "auprc": 0.5503940129055307,
    "paired_win_rate": 0.7457142857142857
  }
}
